# 센서(env) 데이터 시각화 EDA

01_eda.py가 텍스트/숫자 기반 분석이라면, 이 노트북은 **시각화 전용**입니다 (상관관계 히트맵, 분포 히스토그램/박스플롯, 액추에이터 작동비율, 시차(lag) 상관관계). 결과 이미지는 `eda_outputs/`에 저장됩니다. 전체 핵심 인사이트 요약은 README.md, 진행상황은 PLAN.md 참고.

In [ ]:
"""
온실(Greenhouse) 센서 데이터 - 기본 EDA (탐색적 데이터 분석)
파일 경로만 본인 환경에 맞게 수정해서 사용하세요.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 깨짐 방지 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# =========================================
# 1. 데이터 불러오기
# =========================================
df = pd.read_csv('dataset/train/env/train_X.csv')  # 본인 파일 경로로 수정
print("데이터 shape:", df.shape)
df.head()


# =========================================
# 2. 기본 정보 확인
# =========================================
print("\n--- 데이터 타입 및 결측치 ---")
df.info()

print("\n--- 결측치 개수 ---")
print(df.isnull().sum())

print("\n--- 기술통계 요약 ---")
df.describe()


# =========================================
# 3. time 컬럼을 datetime으로 변환 (시계열 처리 필수!)
# =========================================
# 'DAT109 00:00' 같은 형식이면 우선 그대로 순서(index)만 활용하거나,
# 실제 날짜 형식이 있다면 pd.to_datetime으로 변환
# df['time'] = pd.to_datetime(df['time'])
# df = df.set_index('time')


# =========================================
# 4. 상관관계 히트맵
# =========================================
plt.figure(figsize=(14, 10))
corr = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('변수 간 상관관계 히트맵')
plt.tight_layout()
plt.savefig('eda_outputs/correlation_heatmap.png', dpi=150)
plt.show()

# target(예: 내부 temperature)과 상관 높은 변수만 따로 확인
target = 'temperature'
print(f"\n--- {target}와의 상관관계 (절댓값 기준 정렬) ---")
print(corr[target].abs().sort_values(ascending=False))


# =========================================
# 5. 시계열 라인 플롯 (외부 vs 내부 온도 비교)
# =========================================
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(df.index, df['temperature_outside'], label='외부온도', color='tab:blue')
axes[0].plot(df.index, df['temperature'], label='내부온도', color='tab:red')
axes[0].legend()
axes[0].set_title('외부 vs 내부 온도 변화')

axes[1].plot(df.index, df['humidity_outside'], label='외부습도', color='tab:blue')
axes[1].plot(df.index, df['humidity'], label='내부습도', color='tab:red')
axes[1].legend()
axes[1].set_title('외부 vs 내부 습도 변화')

axes[2].plot(df.index, df['co2'], label='CO2', color='tab:green')
axes[2].legend()
axes[2].set_title('CO2 농도 변화')

plt.tight_layout()
plt.savefig('eda_outputs/timeseries_comparison.png', dpi=150)
plt.show()


# =========================================
# 6. 분포 확인 (히스토그램 + 박스플롯으로 이상치 체크)
# =========================================
numeric_cols = ['temperature_outside', 'humidity_outside', 'solar_radiation',
                 'wind_speed_outside', 'temperature', 'humidity', 'co2']

fig, axes = plt.subplots(2, len(numeric_cols), figsize=(20, 6))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i])
    axes[0, i].set_title(col)
    sns.boxplot(y=df[col], ax=axes[1, i])
plt.tight_layout()
plt.savefig('eda_outputs/distribution_boxplot.png', dpi=150)
plt.show()


# =========================================
# 7. 액추에이터(제어장치) 작동 비율 확인 (클래스 불균형 체크)
# =========================================
actuator_cols = ['greenhouse_roof_vent1', 'greenhouse_roof_vent2', 'shading_curtain',
                  'thermal_curtain', 'fcu_fan', 'fcu_pump', 'circ_fan',
                  'co2_supply', 'tube_rail_valve', 'fogging']

on_ratio = (df[actuator_cols] > 0).mean().sort_values(ascending=False)
print("\n--- 각 장치의 작동(ON) 비율 ---")
print(on_ratio)

plt.figure(figsize=(10, 5))
on_ratio.plot(kind='bar', color='teal')
plt.ylabel('작동 비율')
plt.title('액추에이터별 작동 비율')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_outputs/actuator_on_ratio.png', dpi=150)
plt.show()


# =========================================
# 8. 시차(lag) 상관관계 확인 (외부 온도가 내부에 반영되기까지 지연 있는지)
# =========================================
max_lag = 30  # 몇 분까지 확인할지
lag_corrs = []
for lag in range(0, max_lag + 1):
    shifted = df['temperature_outside'].shift(lag)
    lag_corrs.append(df['temperature'].corr(shifted))

plt.figure(figsize=(10, 5))
plt.plot(range(0, max_lag + 1), lag_corrs, marker='o')
plt.xlabel('Lag (분)')
plt.ylabel('상관계수')
plt.title('외부온도 → 내부온도 시차 상관관계')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('eda_outputs/lag_correlation.png', dpi=150)
plt.show()

best_lag = int(np.argmax(lag_corrs))
print(f"\n가장 상관관계가 높은 시차: {best_lag}분 (상관계수: {lag_corrs[best_lag]:.3f})")